In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))

In [ ]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import drawImageAndFeatureMasks
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
from Libraries.inference_training import createModelInstance, testInference
from Libraries.inference_training import testInferenceWithIoU
from paths import MASKRCNN_RESEARCH_TRAIN, MASKRCNN_RESEARCH_TEST

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [ ]:
# train on the GPU or on the CPU, if a GPU is not available
config = Configuration()
print("Device: " + str(config.device))

config.setDatasetPaths(trainPath=MASKRCNN_RESEARCH_TRAIN, testPath=MASKRCNN_RESEARCH_TEST)
config.setFilePrefix("")
config.setModelName("mytrainedmodel")
config.setInputSizes(inputWidth=250, inputHeight=250)
config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)

config.setVersion(20250121)

print("Version: " + str(config.version))

config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                    bboxOverlap=True, bboxPerImage=250, reuseModel=False)
config.setEpochs(5)

description = "Model description"
config.setOnnxInfo(producer="Tygron", description=description)

config.addLegendEntry("Background", 0, "#00000000")
config.addLegendEntry("Label name 1", 1, "#00ffbf")
config.addLegendEntry("Label name 2", 2, "#12d900")

config.setOnnxMetaData(scoreThreshold=0.2,
                       maskThreshold=0.3,
                       strideFraction=0.5)

config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
trainingDataset = ImageDataset(config, True, createTransforms(True))
testDataset = ImageDataset(config, False, createTransforms(False))

print("Train Image count: "+str(trainingDataset.__len__()))
print("Test Image count: "+str(testDataset.__len__()))

if not trainingDataset.validateFiles(False):
    print("Inconsistent training dataset ")
    trainingDataset.validateFiles(True)

if not testDataset.validateFiles(False):
    print("Inconsistent test dataset ")
    testDataset.validateFiles(True)

print("Pytorch model name " + config.getPytorchModelFileName())
print("Onnx file name " + config.getOnnxFileName())

In [ ]:
imageNumber = 5
print(trainingDataset.getLabelList(imageNumber))
drawImageAndFeatureMasks(config, trainingDataset, imageNumber)

In [ ]:
loadExistingModel = False

if loadExistingModel:
    model = createModelInstance(config)
    loadModel(config, model, path=config.getPytorchModelFileName())

else:
    model = trainModel(config, trainingDataset, testDataset)
    saveModel(config, model, path=config.getPytorchModelFileName())

In [ ]:
model.eval()
testPrediction = testInference(config, model=model,
                               dataset=testDataset, imageNumber=88)

In [ ]:
exportOnnxModel(config, model)

In [ ]:
writeONNXMeta(config)

In [ ]:
onnx_model = loadONNX(config)
print(f"metadata_props={onnx_model.metadata_props}")